<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/Project/update_model_df.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df_v1 = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/model_data_v1.feather')

In [4]:
# Drop discharge year
df_v2 = df_v1.copy()
df_v2.drop(columns=['discharge_year'], inplace=True)

In [5]:
for payer in df_v2.columns[df_v2.columns.str.contains('payer')]:
  print(payer)

payer_medicaid
payer_medicare
payer_private_insurance
payer_self_pay
payer_blue_cross
payer_other
payer_gov_va
payer_corrections
payer_managed_care


In [6]:
# Combine payer cols to major payer types (based on feature importance)

def map_payment_types(row):
  payers = []

  # Major Categories
  # 1. Medicare
  if row.get('payer_medicare', 0) == 1:
    payers.append('Medicare')

  # 2. Medicaid
  if row.get('payer_medicaid', 0) == 1:
    payers.append('Medicaid')

  # 3. Private insurance
  if row.get('payer_private_insurance', 0) == 1:
    payers.append('Private insurance')

  # 4. BlueCross
  if row.get('payer_blue_cross', 0) == 1:
    payers.append('BlueCross')

  # 5. Everything else -> Other
  if (
      row.get('payer_managed_care', 0) == 1 or
      row.get('payer_self_pay', 0) == 1 or
      row.get('payer_corrections', 0) == 1 or
      row.get('payer_gov_va', 0) == 1 or
      row.get('payer_other', 0) == 1
  ):
    payers.append('Other')

  if len(payers) == 0:
    return 'Unknown'

  return ', '.join(sorted(payers))

In [7]:
# Apply payer changes to df
df_v2['payment_type'] = df_v2.apply(map_payment_types,axis=1)
df_v2['payment_type'] = df_v2['payment_type'].astype('object')

In [8]:
# Drop old payer cols
payer_cols = [
    'payer_medicaid',
    'payer_medicare',
    'payer_private_insurance',
    'payer_self_pay',
    'payer_blue_cross',
    'payer_other',
    'payer_gov_va',
    'payer_corrections',
    'payer_managed_care'
]
df_v2.drop(columns=payer_cols, inplace=True)

In [9]:
df_v2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238636 entries, 0 to 4238635
Data columns (total 19 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   health_service_area  object
 1   hospital_county      object
 2   facility_id          object
 3   age_group            object
 4   zip_code             object
 5   gender               object
 6   race                 object
 7   ethnicity            object
 8   length_of_stay       int64 
 9   admission_type       object
 10  ccsr_dx_code         object
 11  ccsr_px_code         object
 12  apr_drg_code         object
 13  apr_mdc_code         object
 14  apr_severity_code    object
 15  apr_mortality_risk   object
 16  apr_med_surg_desc    object
 17  num_payment_types    int64 
 18  payment_type         object
dtypes: int64(2), object(17)
memory usage: 614.4+ MB


In [10]:
df_v2.value_counts('payment_type')

,count
payment_type,
Medicaid,1037669
Medicare,561642
"Medicaid, Medicare",549050
Private insurance,468121
BlueCross,297818
"Medicare, Private insurance",275686
"Medicaid, Other",219943
"BlueCross, Medicare",200650
Other,170353


In [12]:
df_v2.to_feather('/content/drive/MyDrive/Sparcs_Datafiles/model_data_v2.feather')